# Evaluation Inspection

Explore RAG evaluation: run the answer pipeline on gold Q&A, compare with reference answers using **exact match**, **contains** (ref-in-pred), and optionally **LLM-as-judge**; report accuracy.

**Goals:**
- Load `data/eval/questions.json` (handbook Q&A pairs)
- For each question: retrieve → `answer(question, chunks)` → compare `pred` to `ref`
- Implement `exact_match`, `ref_in_pred`; optionally `llm_judge`
- Report per-question and overall % correct to tune chunk size, top_k, or prompt

## 1. Setup

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from src.config.settings import PROJECT_ROOT, CHUNKS_JSONL, INDEX_PATH, TOP_K

QUESTIONS_JSON = PROJECT_ROOT / "data" / "eval" / "questions.json"

print(f"Eval questions: {QUESTIONS_JSON}, exists: {QUESTIONS_JSON.exists()}")
print(f"Index: {INDEX_PATH}, exists: {INDEX_PATH.exists()}")
print(f"Chunks: {CHUNKS_JSONL}, exists: {CHUNKS_JSONL.exists()}")
print(f"TOP_K: {TOP_K}")

Eval questions: /Users/srujayreddy/Projects/ca-dmv-rag-system/data/eval/questions.json, exists: True
Index: /Users/srujayreddy/Projects/ca-dmv-rag-system/data/embeddings/handbook.index, exists: False
Chunks: /Users/srujayreddy/Projects/ca-dmv-rag-system/data/processed/chunks.jsonl, exists: True
TOP_K: 5


## 2. Load eval questions

In [2]:
with open(QUESTIONS_JSON) as f:
    questions = json.load(f)
print(f"Loaded {len(questions)} Q&A pairs")

Loaded 15 Q&A pairs


## 3. Build retriever

In [3]:
from src.retrieval import Retriever, embed_query, embed_texts
from src.retrieval.vector_store import VectorStore

if INDEX_PATH.exists():
    retriever = Retriever(INDEX_PATH)
    print("Using persisted index:", INDEX_PATH)
else:
    if not CHUNKS_JSONL.exists():
        raise FileNotFoundError(f"Need {CHUNKS_JSONL}. Run: python scripts/build_chunks.py")
    chunks_list = []
    with open(CHUNKS_JSONL) as f:
        for line in f:
            if line.strip():
                chunks_list.append(json.loads(line))
    texts = [c["text"] for c in chunks_list]
    embs = embed_texts(texts)
    _store = VectorStore()
    _store.add(embs, chunks_list)
    class _InMemoryRetriever:
        def retrieve(self, q, top_k=5):
            return _store.search(embed_query(q), top_k=top_k)
    retriever = _InMemoryRetriever()
    print("(Using in-memory index from chunks.jsonl. Run: python scripts/build_index.py to persist)")

/Users/srujayreddy/Projects/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


(Using in-memory index from chunks.jsonl. Run: python scripts/build_index.py to persist)


## 4. Metrics

In [4]:
import re

def _normalize(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def exact_match(pred: str, ref: str) -> bool:
    """True if normalized prediction equals normalized reference."""
    return _normalize(pred) == _normalize(ref)

def ref_in_pred(pred: str, ref: str) -> bool:
    """True if the reference (or its key parts) appears in the prediction (case-insensitive)."""
    # For short refs, check if ref is a substring of pred.
    # For longer refs, check if ref is contained (handles reordering).
    np, nr = _normalize(pred), _normalize(ref)
    if not nr:
        return True
    return nr in np

## 5. Eval loop

In [5]:
from openai import AuthenticationError, NotFoundError, RateLimitError
from src.generation import answer

results = []
for item in questions:
    qid, q, ref = item["id"], item["question"], item["answer"]
    chunks = retriever.retrieve(q, top_k=TOP_K)
    try:
        pred = answer(q, context_chunks=chunks)
    except (ValueError, AuthenticationError, NotFoundError, RateLimitError) as e:
        pred = ""
        print(f"[{qid}] Error: {e}")
    em = exact_match(pred, ref)
    rp = ref_in_pred(pred, ref)
    results.append({"id": qid, "question": q, "ref": ref, "pred": pred, "exact_match": em, "ref_in_pred": rp})

print(f"Ran eval on {len(results)} questions.")

Ran eval on 15 questions.


## 6. Report

In [6]:
n = len(results)
em_correct = sum(1 for r in results if r["exact_match"])
rp_correct = sum(1 for r in results if r["ref_in_pred"])

print("Per-question:")
print("id  exact  ref_in_pred  question (truncated)")
for r in results:
    print(f"{r['id']:>2}  {r['exact_match']!s:>5}  {r['ref_in_pred']!s:>11}  {r['question'][:50]}...")

print()
print(f"Overall: exact_match={em_correct}/{n} ({100*em_correct/n:.1f}%); ref_in_pred={rp_correct}/{n} ({100*rp_correct/n:.1f}%)")

Per-question:
id  exact  ref_in_pred  question (truncated)
 1  False        False  What is the blood alcohol limit for DUI?...
 2  False        False  When must I use headlights?...
 3  False        False  What is the speed limit on most California highway...
 4  False        False  How do I renew my driver license?...
 5  False        False  Within how many feet must I dim my high beams for ...
 6  False        False  How soon must I notify DMV when I change my addres...
 7  False        False  How old must you be to get an instruction permit i...
 8  False        False  What are the provisional license restrictions for ...
 9  False         True  How far in advance should you signal before turnin...
10  False        False  When may you drive in a bicycle lane?...
11  False        False  How long may you drive in a center left turn lane?...
12  False        False  When must you use a turnout to let vehicles pass?...
13  False        False  May you pass over double solid yellow lines?.

## 7. (Optional) LLM-as-judge

In [7]:
from src.generation import generate

def llm_judge(question: str, ref: str, pred: str) -> bool:
    prompt = f"""You are a judge. Given:
Question: {question}
Reference answer: {ref}
Model prediction: {pred}

Does the prediction contain the key information from the reference? Be lenient: if the main facts are present, say YES. Reply with only YES or NO."""
    out = generate(prompt).strip().upper()
    return out.startswith("YES")

# Run LLM judge on first 3 for a quick check (optional; uses extra API calls)
for r in results[:3]:
    try:
        j = llm_judge(r["question"], r["ref"], r["pred"])
        print(f"id={r['id']} llm_judge={j}  Q: {r['question'][:45]}...")
    except Exception as e:
        print(f"id={r['id']} llm_judge error: {e}")

id=1 llm_judge=True  Q: What is the blood alcohol limit for DUI?...
id=2 llm_judge=True  Q: When must I use headlights?...
id=3 llm_judge=True  Q: What is the speed limit on most California hi...


## 8. Summary

In [8]:
print("Evaluation pipeline:")
print("  - Load data/eval/questions.json")
print("  - Retriever.retrieve(question, top_k) -> answer(question, chunks) -> pred")
print("  - exact_match(pred, ref), ref_in_pred(pred, ref); optional llm_judge(question, ref, pred)")
print("  - Use scripts/run_eval.py to re-run and tune chunk size, top_k, or prompt.")

Evaluation pipeline:
  - Load data/eval/questions.json
  - Retriever.retrieve(question, top_k) -> answer(question, chunks) -> pred
  - exact_match(pred, ref), ref_in_pred(pred, ref); optional llm_judge(question, ref, pred)
  - Use scripts/run_eval.py to re-run and tune chunk size, top_k, or prompt.
